In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import joblib
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
Sequential = tf.keras.models.Sequential
Embedding = tf.keras.layers.Embedding
LSTM = tf.keras.layers.LSTM
Dense =  tf.keras.layers.Dense
Dropout =  tf.keras.layers.Dropout
SpatialDropout1D = tf.keras.layers.SpatialDropout1D
sequence = tf.keras.preprocessing.sequence
from sklearn.preprocessing import LabelEncoder
print("Downloading NLTK data...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
try:
    df = pd.read_csv("twitter_training.csv")
except FileNotFoundError:
    print("Error: 'twitter_training.csv' not found.")
    df = pd.DataFrame({
        'text': ["I love this product", "Terrible service", "Great quality", "Waste of money"],
        'label': [1, 0, 1, 0]
    })
print("Columns:", df.columns.tolist())
text_col = "text"
target_col = "label"    
print(f"Using Text Column: '{text_col}' | Target Column: '{target_col}'")
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def clean_and_tokenize(text):
    if pd.isna(text): return []
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+|#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens
print("Cleaning text...")
df['tokens'] = df[text_col].apply(clean_and_tokenize)
df['cleaned_text'] = df['tokens'].apply(lambda x: ' '.join(x))
df = df.drop(columns=[text_col])
print("Generating TF-IDF features...")
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf = tfidf_vec.fit_transform(df['cleaned_text'])
print("Training Word2Vec model...")
token_list = df['tokens'].tolist()
w2v_model = Word2Vec(sentences=token_list, vector_size=100, window=5, min_count=2, workers=4, epochs=20)
w2v_model.save('word2vec_twitter.model')
word_index = {w: i+1 for i, w in enumerate(w2v_model.wv.index_to_key)} 
max_len = 50 
def tokens_to_padded_sequence(tokens):
    seq = [word_index.get(t, 0) for t in tokens]
    if len(seq) > max_len:
        seq = seq[:max_len]
    if len(seq) < max_len:
        seq = [0] * (max_len - len(seq)) + seq
    return seq
print("Creating padded sequences for LSTM...")
df['seq'] = df['tokens'].apply(tokens_to_padded_sequence)
X_lstm = np.array(df['seq'].tolist())
# LABEL ENCODING  Convert string labels (e.g., "positive") to integers (e.g., 0, 1)
le = LabelEncoder()
y_encoded = le.fit_transform(df[target_col])
print(f"Classes found: {le.classes_}")
print(f"Encoded labels sample: {y_encoded[:5]}")
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, y_encoded, test_size=0.2, random_state=42)
X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(X_lstm, y_encoded, test_size=0.2, random_state=42)
# Model 1: Naïve Bayes
print("\n--- Training Naïve Bayes ---")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)
print(f"NB Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
# Model 2: Logistic Regression
print("\n--- Training Logistic Regression ---")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)
print(f"LR Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
# Model 3: LSTM (Deep Learning)
print("\n--- Training LSTM ---")
vocab_size = len(word_index) + 1
embedding_dim = 100
num_classes = len(le.classes_)
# Build an embedding matrix from the trained Word2Vec vectors so the LSTM , starts from those embeddings instead of random ones.
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim,weights=[embedding_matrix], input_length=max_len, trainable=True),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax') 
])
lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()
lstm_model.fit(X_train_lstm, y_train_lstm, 
               epochs=50, 
               batch_size=64, 
               validation_data=(X_test_lstm, y_test_lstm))
y_pred_lstm_prob = lstm_model.predict(X_test_lstm)
y_pred_lstm = np.argmax(y_pred_lstm_prob, axis=1)
print(f"LSTM Accuracy: {accuracy_score(y_test_lstm, y_pred_lstm):.4f}")
joblib.dump(nb_model, 'nb_model.pkl')
joblib.dump(lr_model, 'lr_model.pkl')
joblib.dump(tfidf_vec, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
lstm_model.save('lstm_model.h5')
print("\nAll models saved successfully!")

Columns: ['text', 'label']
Using Text Column: 'text' | Target Column: 'label'
Cleaning text...
Generating TF-IDF features...
Training Word2Vec model...
Creating padded sequences for LSTM...
Classes found: ['negative' 'neutral' 'positive']
Encoded labels sample: [1 2 0 0 1]

--- Training Naïve Bayes ---
NB Accuracy: 1.0000

--- Training Logistic Regression ---
LR Accuracy: 1.0000

--- Training LSTM ---


c:\Users\faiza\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │        14,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_7             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,200 (55.47 KB)

 Trainable params: 14,200 (55.47 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.3483 - loss: 1.1043 - val_accuracy: 0.4533 - val_loss: 1.0820
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.4167 - loss: 1.0805 - val_accuracy: 1.0000 - val_loss: 1.0577
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5033 - loss: 1.0443 - val_accuracy: 0.7600 - val_loss: 1.0010
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6133 - loss: 0.9698 - val_accuracy: 0.9667 - val_loss: 0.8597
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7300 - loss: 0.8172 - val_accuracy: 0.9933 - val_loss: 0.5848
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8467 - loss: 0.5533 - val_accuracy: 1.0000 - val_loss: 0.2747
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9433 - loss: 0.3075 - val_accuracy: 1.0000 - val_loss: 0.0714
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9650 - loss: 0.1603 - val_accuracy: 1.0000 - v

LSTM Accuracy: 1.0000

All models saved successfully!


In [14]:
import json
import pickle
import numpy as np
import pandas as pd
import gradio as gr
import torch
from preprocess import preprocess_text, preprocess_to_string
from train import SentimentLSTM, encode_and_pad

ARTIFACT_DIR = "artifacts"

print("Loading artifacts...")
with open(f"{ARTIFACT_DIR}/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
with open(f"{ARTIFACT_DIR}/naive_bayes.pkl", "rb") as f:
    nb_model = pickle.load(f)
with open(f"{ARTIFACT_DIR}/logistic_regression.pkl", "rb") as f:
    lr_model = pickle.load(f)
with open(f"{ARTIFACT_DIR}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
with open(f"{ARTIFACT_DIR}/word2idx.json", "r") as f:
    word2idx = json.load(f)
with open(f"{ARTIFACT_DIR}/summary.json", "r") as f:
    summary = json.load(f)

checkpoint = torch.load(f"{ARTIFACT_DIR}/lstm_model.pt", map_location="cpu")
vocab_size, embed_dim = checkpoint["embedding_matrix_shape"]
dummy_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)
lstm_model = SentimentLSTM(dummy_matrix, num_classes=checkpoint["num_classes"])
lstm_model.load_state_dict(checkpoint["model_state"])
lstm_model.eval()
max_len = checkpoint["max_len"]
print("Artifacts loaded.")

EMOJI = {"positive": "😊", "negative": "😠", "neutral": "😐"}


def predict_batch(texts, model_choice):
    if model_choice in ("Naive Bayes", "Logistic Regression"):
        cleaned = [preprocess_to_string(t) for t in texts]
        X = vectorizer.transform(cleaned)
        model = nb_model if model_choice == "Naive Bayes" else lr_model
        preds = model.predict(X)
        return list(label_encoder.inverse_transform(preds))
    # LSTM
    tokenized = [preprocess_text(t) for t in texts]
    X, lengths = encode_and_pad(tokenized, word2idx, max_len)
    with torch.no_grad():
        logits = lstm_model(X, lengths)
        preds = logits.argmax(dim=1).numpy()
    return list(label_encoder.inverse_transform(preds))


def analyze_single(text, model_choice):
    if not text.strip():
        return "Please enter some text."
    label = predict_batch([text], model_choice)[0]
    emoji = EMOJI.get(label, "")
    return f"Predicted sentiment: **{label.upper()}** {emoji}"


def analyze_csv(file, model_choice):
    if file is None:
        return "Please upload a CSV.", None, None, None

    df = pd.read_csv(file.name)
    if "text" not in df.columns:
        return "CSV must contain a 'text' column.", None, None, None

    df["predicted_sentiment"] = predict_batch(df["text"].astype(str).tolist(), model_choice)

    counts = df["predicted_sentiment"].value_counts(normalize=True).reindex(
        ["positive", "neutral", "negative"]
    ).fillna(0) * 100
    dist_df = pd.DataFrame({"sentiment": counts.index, "percentage": counts.values})

    out_path = "sentiment_results.csv"
    df.to_csv(out_path, index=False)

    summary_text = (
        f"Positive: {counts.get('positive', 0):.1f}%  |  "
        f"Neutral: {counts.get('neutral', 0):.1f}%  |  "
        f"Negative: {counts.get('negative', 0):.1f}%"
    )
    return summary_text, dist_df, df[["text", "predicted_sentiment"]], out_path


with gr.Blocks(title="Sentiment Dashboard") as demo:
    gr.Markdown("# 💬 Public Sentiment Dashboard")
    gr.Markdown("TF-IDF + Naive Bayes / Logistic Regression, and Word2Vec + LSTM")

    model_choice = gr.Radio(
        ["Logistic Regression", "Naive Bayes", "LSTM"],
        value="Logistic Regression",
        label="Model",
    )
    with gr.Accordion("Last training run", open=False):
        gr.JSON(summary)

    with gr.Tab("Single text"):
        text_input = gr.Textbox(
            label="Enter a tweet / review to analyze",
            placeholder="e.g. This product completely exceeded my expectations!",
            lines=3,
        )
        analyze_btn = gr.Button("Analyze", variant="primary")
        single_result = gr.Markdown()
        analyze_btn.click(analyze_single, [text_input, model_choice], single_result)

    with gr.Tab("Batch CSV"):
        file_input = gr.File(label="Upload a CSV with a 'text' column", file_types=[".csv"])
        batch_btn = gr.Button("Analyze", variant="primary")
        batch_summary = gr.Markdown()
        batch_chart = gr.BarPlot(x="sentiment", y="percentage", title="Overall sentiment breakdown")
        batch_table = gr.Dataframe(label="Predictions")
        download = gr.File(label="Download results as CSV")
        batch_btn.click(
            analyze_csv,
            [file_input, model_choice],
            [batch_summary, batch_chart, batch_table, download],
        )

if __name__ == "__main__":
    demo.launch()

Loading artifacts...
Artifacts loaded.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
